# 05: Random Forest Classifier

**Goal:** Train random forest for non-linear feature interactions and compare to linear models.

## Key Deliverable
Top-30 RF feature importances vs. LASSO agreement

In [ ]:
import sys
sys.path.insert(0, "../")

import pandas as pd
import numpy as np
from src import data_utils, nlp_utils, model_utils, viz_utils

SEED = 42
np.random.seed(SEED)
import random
random.seed(SEED)

print("Imports successful!")

## Step 1: Load Data

In [ ]:
df = pd.read_csv("../data/processed/bills_speeches_preprocessed.csv")
X_text = df['speeches_combined'].values
y = df['passed'].values

# Create TF-IDF features
X_tfidf, vectorizer, feature_names = nlp_utils.create_tfidf_features(
    X_text,
    max_features=5000,
    min_df=5,
    max_df=0.95,
)

print(f"Data loaded: {len(df)} bills, {X_tfidf.shape[1]} TF-IDF features")

## Step 2: Train Random Forest

In [ ]:
# Convert sparse TF-IDF to dense for Random Forest
X_dense = X_tfidf.toarray()

# Train Random Forest
rf_model = model_utils.train_random_forest(
    X_dense, y,
    n_estimators=200,
    max_depth=None,
    random_state=SEED
)

print("✓ Random Forest trained with 200 trees")

## Step 3: Extract Feature Importances

In [ ]:
# Get top-30 RF features
top_rf = model_utils.get_top_features_rf(rf_model, feature_names, top_n=30)
print("\nTop-30 Random Forest Features:")
print(top_rf)

# Save
top_rf.to_csv("../results/tables/rf_top_features.csv", index=False)
print("\nSaved to results/tables/rf_top_features.csv")

In [ ]:
# Plot RF importances
viz_utils.plot_feature_importance(
    top_rf,
    output_path="../results/figures/rf_importance.png",
    title="Random Forest Feature Importances (Top-30)",
    max_features=30
)

## Step 4: Model Evaluation

In [ ]:
# Evaluate with CV
evaluator = model_utils.ModelEvaluator(random_state=SEED)
rf_results = evaluator.evaluate_classifier(
    rf_model, X_dense, y, model_name="Random Forest", cv_splits=5
)

print("\n=== Random Forest Performance ===")
for key, value in rf_results.items():
    if not key == "model":
        print(f"{key}: {value:.4f}")

# Save
results_df = pd.DataFrame([rf_results])
results_df.to_csv("../results/tables/rf_cv_scores.csv", index=False)

## Step 5: LASSO vs RF Feature Comparison

In [ ]:
# Load LASSO features
lasso_features = pd.read_csv("../results/tables/lasso_top_features.csv")

# Compare top-20 features
lasso_top20 = set(lasso_features['feature'].head(20))
rf_top20 = set(top_rf['feature'].head(20))

overlap = lasso_top20 & rf_top20

print(f"\nLASSO top-20 features: {len(lasso_top20)}")
print(f"RF top-20 features: {len(rf_top20)}")
print(f"Overlap: {len(overlap)} features")
print(f"\nOverlapping features:")
for feat in sorted(overlap):
    print(f"  - {feat}")

**Next:** Run `06_bert_classifier.ipynb` (optional) or proceed to `07_results_comparison.ipynb`